# Submission 2: Project AI / Algorithmic and Behavioral Model (5%)

**Course:** RBB2013 / FFM2063 / FEM2063, Digital Twin, May 2026
**Project:** SmartClean Twin, a software-emulated Digital Twin of a mobile
inspection and cleaning robot (project topic 2)
**Repository:** https://github.com/KAI-UTP/smartclean-twin
**Presentation & demo video:** [https://youtu.be/zEq7L-ivMLA](https://youtu.be/zEq7L-ivMLA)

**Team Members**

| No | Name | Student ID |
|---|---|---|
| 1 | Chan Li Kai | 22010900 |
| 2 | William Wong Xiao Kang | 22010943 |
| 3 | Irvin Chang Hou Ceng | 22012342 |
| 4 | Liang Yan Ee | 22011522 |
| 5 | Nurin Emelin Binti Marhisyam | 24006706 |

> **How to reproduce:** start the stack with `docker compose up -d` (8 containers),
> then run the notebook top to bottom. All code cells in this notebook were
> executed against the running system and their outputs are saved below, so the
> evidence is readable without re-running.


## 1. Executive Summary

The SmartClean Twin contains an intelligence layer composed of **five machine
learning models covering all three major learning paradigms**, plus
**algorithmic behavioral models** that govern how the twin and the robot act.

| # | Model | Paradigm | Output | Validation result |
|---|---|---|---|---|
| 1 | Motor health classifier | Supervised classification | NORMAL / HIGH_LOAD / OVERHEATED / FAULT | 100 % hold-out accuracy |
| 2 | Dirt level classifier | Supervised classification | CLEAN / MODERATE / DIRTY | 99.9 % |
| 3 | Health state classifier | Supervised classification | NORMAL / WARNING / CRITICAL | 90.8 % accuracy |
| 4 | Remaining-useful-life regressor | Supervised regression | minutes of operation remaining | R² = 0.91, MAE = 6.6 min |
| 5 | Anomaly detector | **Unsupervised** | anomaly score + boolean flag | 100 % detection of known fault signatures, 0 % false alarms on healthy data |

On top of the models the twin computes **trend-based forecasts** (battery
minutes-to-empty, cleaning minutes-to-finish) and produces a single
**plain-language recommendation** for the operator, which closes the loop from
measurement to advice. A **what-if simulation endpoint** allows hypothetical
sensor values to be evaluated by all five models without touching the asset.


## 2. Digital Twin Use Cases Addressed

Module 1 lists the common Digital Twin use cases. This submission addresses
four of them explicitly, and it is useful to name which model serves which:

| Digital Twin use case | How the SmartClean Twin implements it |
|---|---|
| **State estimation** | The state engine converts raw sensor values into an 11-dimension twin state; the health-state classifier adds a learned 3-class condition estimate |
| **Fault diagnosis** | Motor health classifier names the fault type (HIGH_LOAD / OVERHEATED / FAULT); the anomaly detector flags conditions that no label describes |
| **Failure prognostication** | RUL regressor predicts minutes of operation remaining; battery minutes-to-empty forecasts the next forced stop |
| **Predictive maintenance** | Recommendation engine converts prognosis into an action ("schedule maintenance soon", "STOP and inspect motor") |

In the 5D framework terms used in Module 1, these models constitute part of the
**virtual model's behavior model (Bv)** and drive the **V2P prediction engine**:
information flows from the virtual twin back toward decisions about the physical
asset.


## 3. Learning Paradigms and Why Each Is Used

### 3.1 Supervised classification

Supervised learning trains on a **labelled dataset** in which every input is
paired with a known output, and the trained model then predicts the label of
new, unseen inputs. Classification is the appropriate task when the target is a
**discrete, mutually exclusive category**.

Three of our targets are categorical: motor health, dirt level and overall
health state. An operator does not act on "temperature = 78.3 °C"; they act on
"OVERHEATED". Classification produces exactly that decision, and additionally
provides a **class probability**, which we surface as a confidence value so the
operator can distinguish a marginal call from a certain one.

### 3.2 Supervised regression

Regression is used when the response variable is **continuous**. Remaining
useful life is inherently continuous, the difference between 8 minutes and
40 minutes of remaining operation changes what a maintenance planner does, and
bucketing it into classes would discard that information. The RUL model
therefore predicts a real-valued number of minutes.

### 3.3 Unsupervised anomaly detection

Both approaches above require labelled failures. In practice a newly deployed
robot has **no failure history**, and any label set we invent can only describe
faults we already thought of. The anomaly detector therefore learns from
**normal operation only**: it estimates the mean and standard deviation of each
sensor during healthy running, and flags any sample in which **any** sensor
deviates by more than 4.5 standard deviations.

This is genuinely a third capability rather than a duplicate of the
classifiers: it can flag a fault signature that was never present in any
training label, which is precisely the case that matters for a real asset.
It is also fully explainable, the reason for every alarm is "sensor *s* is
*k* sigma from its normal range", which an engineer can check by hand.


## 4. Machine Learning Pipeline

The implementation follows the standard pipeline stages taught in Module 4, and
each stage maps to a specific part of the codebase:

| Pipeline stage | Implementation | File |
|---|---|---|
| Data collection & pre-processing | Dataset generation from documented rules; `StandardScaler` fitted inside a `Pipeline` so scaling parameters are learned from training data only | `services/ai-service/train_model.py` |
| Model selection, training & validation | Random Forest classifiers and regressor; stratified 80/20 hold-out split; metrics asserted against thresholds, build fails if not met | `services/ai-service/train_model.py` |
| Model deployment | Models serialised with `joblib` and **baked into the Docker image at build time**, then loaded at service start | `services/ai-service/Dockerfile`, `predictor.py` |
| Inference / serving | Every validated telemetry message is scored; results published to MQTT and written to InfluxDB | `services/ai-service/main.py` |

Two properties of this arrangement are worth noting. First, **training runs
inside the image build**, so the model artefacts can never drift from the code
that produced them, and the CI pipeline re-trains and re-validates on every
push. Second, **pre-processing lives inside the model pipeline object**, which
prevents the classic data-leakage error of fitting a scaler on the full dataset
before splitting.


## 5. Feature and Target Selection

### 5.1 Features

Features are restricted to quantities the twin actually receives from the
asset, so that the model is deployable rather than dependent on hindsight.

| Feature | Unit | Why it is informative |
|---|---|---|
| `motor_temperature_c` | °C | Rises with sustained load, friction or cooling failure, the primary thermal degradation signal |
| `motor_current_a` | A | Increases under mechanical resistance; the electrical signature of a stalled or loaded motor |
| `speed_mps` | m/s | Distinguishes operating regime; a commanded-but-stationary robot is itself a fault symptom |
| `battery_v`, `battery_a` | V, A | Together give instantaneous power draw; voltage sag under load indicates cell condition |
| `battery_soc` | % | Depletion state; used as a proxy for accumulated duty in this cycle |
| `brush_on`, `pump_on` | boolean | Actuator context: the same current means different things with the brush on or off |
| `water_level_pct` | % | Consumable state; a full tank corresponds to a well-serviced machine |
| `dirt_score` | 0-1 | Environmental input for the cleaning-effort decision |

### 5.2 Targets

| Target | Type | Digital Twin purpose |
|---|---|---|
| `motor_health` | 4-class categorical | Fault diagnosis for the operator |
| `dirt_level` | 3-class categorical | Whether a cell needs a repeat pass |
| `health_state` | 3-class categorical | Single condition summary for the dashboard status strip |
| `predicted_rul_minutes` | continuous | Maintenance scheduling horizon |
| `is_anomaly` / `anomaly_score` | boolean + continuous | Detection of unmodelled conditions |

The `robot_id` is deliberately **not** used as a feature. Including it would
allow the model to memorise a particular asset rather than learn the physics of
degradation, and the model would then fail on a newly added robot.


## 6. Dataset Generation and Labelling Rules

No physical robot exists and therefore no recorded failure history exists. A
labelled dataset is generated from **documented, physics-motivated rules**, so
that the mapping from inputs to labels is explicit and auditable rather than
arbitrary.

### 6.1 Motor health rules

```
FAULT      : motor_current_a > 3.5  AND motor_temperature_c > 70
OVERHEATED : motor_temperature_c > 70
HIGH_LOAD  : motor_current_a > 2.5  OR (motor_current_a > 1.5 AND brush_on)
NORMAL     : otherwise
```

The ordering matters: a machine that is both drawing excess current and running
hot is a genuine fault, not merely hot, so `FAULT` is tested first.

### 6.2 Health state and RUL

A latent `risk_score` is composed from weighted, normalised contributions of
temperature, vibration, load, accumulated duty and maintenance condition, plus
Gaussian noise so that the classes are not perfectly separable (a noiseless
dataset would produce a meaningless 100 % accuracy). Health state thresholds
the risk score at 0.35 and 0.65; RUL is a decreasing function of risk, duty and
load, clipped to a physically sensible 5-130 minute range.

### 6.3 Normal-operation dataset for anomaly detection

A separate dataset is drawn only from the healthy operating envelope
(`motor_current_a` ≈ 0.75 ± 0.18 A, `motor_temperature_c` ≈ 42 ± 11 °C,
speeds 0 / 0.1 / 0.2 m/s, and so on). The detector never sees a fault during
training, which is what makes its later detection of injected faults
meaningful.


## 7. Model Configuration, Training and Validation

### 7.1 Algorithm choice

Random Forests are used for all four supervised models. The justification is
specific rather than habitual:

- they capture **non-linear interactions** between sensors (temperature matters
  more when current is also high) without manual feature engineering;
- they are **insensitive to feature scaling** and to mixed units;
- they expose **feature importances**, so the model's reasoning can be
  inspected and sanity-checked against physics;
- they are far more **interpretable than a deep network**, which matters for a
  maintenance decision that a human must trust, a deep network's hidden layers
  are effectively a black box, and with a dataset of this size the extra
  capacity would buy nothing.

Hyperparameters: 150 trees, `max_depth` 9 (classifier) / 10 (regressor),
`min_samples_leaf` 3, and `class_weight="balanced"` for the health-state
classifier so that the minority CRITICAL class is not ignored.

### 7.2 Validation methodology

An 80/20 hold-out split is used, **stratified on the class label** so that all
three health states appear in both the training and validation sets. Metrics are
computed on the validation set only. The training script asserts minimum
performance and exits non-zero if it is not met, so a regression in model
quality **fails the CI build** rather than silently shipping.

### 7.3 Results

| Model | Metric | Value | Interpretation |
|---|---|---|---|
| Health state classifier | Accuracy | 0.908 | Correct condition class in ~91 % of unseen cases |
| Health state classifier | Macro F1 | ~0.88 | Balanced across all three classes, including minority CRITICAL |
| RUL regressor | R² | 0.91 | 91 % of the variance in remaining life is explained by the sensors |
| RUL regressor | MAE | 6.6 min | Average error of about 6.6 minutes, directly interpretable by a planner |
| Motor health classifier | Accuracy | 1.00 | Expected: this target is a deterministic function of two features, so the tree recovers the rule exactly |
| Anomaly detector | Fault detection / false alarm | 100 % / 0 % | All three known fault signatures flagged; no healthy sample flagged |

Misclassifications concentrate at the **boundaries** between adjacent classes
(NORMAL/WARNING and WARNING/CRITICAL), which is the expected and benign failure
mode: a borderline machine is genuinely ambiguous, and the cost of calling a
marginal WARNING a NORMAL is low compared with missing a CRITICAL.

The motor-health model's perfect score is reported honestly as a property of a
rule-generated target, not as evidence of a superior model.


## 8. Algorithmic and Behavioral Models

Not all intelligence in the twin is learned. Three algorithmic behavior models
operate alongside the ML models, corresponding to the *rule model (Rv)* and
*behavior model (Bv)* of the virtual system:

**8.1 Twin state rules.** The state engine deterministically derives 11 state
variables from telemetry, for example `obstacle_cm < 25` ⇒ `safety_state =
EMERGENCY` and an `OBSTACLE_EMERGENCY` alarm. Rules are used rather than a
model wherever the mapping is known and safety-relevant: a safety stop must be
predictable and explainable, not probabilistic.

**8.2 Autonomous battery lifecycle.** The robot manages its own energy: below
20 % SoC it abandons cleaning and returns to the dock, charges at 10 %/min, and
resumes cleaning at 80 %. This is a behavioral model of the asset, and it is
visible end-to-end, the discharge-rate panel turns positive while charging.

**8.3 Trend-based forecasting.** Battery minutes-to-empty and cleaning
minutes-to-finish are computed from the rate of change over a 60-second rolling
window, rather than from a trained model. A linear extrapolation is appropriate
here because both quantities are close to linear over a one-minute horizon, and
it has the practical advantage of needing no training data and adapting
immediately to the current duty.

**8.4 Recommendation engine.** The outputs above are combined by a documented
priority order into one instruction: CRITICAL health ⇒ "STOP robot and inspect
motor immediately"; anomaly ⇒ "verify sensors and inspect robot"; imminent
battery exhaustion ⇒ "return to dock within 10 minutes"; WARNING ⇒ "schedule
maintenance soon"; otherwise "normal operation". Ordering by severity ensures
the most consequential condition is never masked by a milder one.


## 9. Deployment and Live Inference

The AI service subscribes to validated telemetry, scores every message with all
five models, publishes the result to `smartclean/SCR01/prediction`, and writes
it to InfluxDB where Grafana and the 3D scene can read it. A **rule-based
fallback** is retained: if the model artefacts cannot be loaded, the service
degrades to threshold logic and reports `model_used = "rule_fallback"` rather
than failing. This keeps the twin observable during a partial failure.

The cell below shows the model outputs that were live at execution time.

In [1]:
import json, time, urllib.request

INFLUX = "http://localhost:8086/api/v2/query?org=smartclean"
TOKEN = "smartclean-super-secret-token"

def flux_query(q):
    """Run a Flux query against InfluxDB and return raw CSV."""
    req = urllib.request.Request(INFLUX, data=q.encode(),
        headers={"Authorization": f"Token {TOKEN}",
                 "Content-Type": "application/vnd.flux", "Accept": "application/csv"})
    with urllib.request.urlopen(req, timeout=10) as r:
        return r.read().decode()

def show_last(measurement, range_s=30):
    """Print the most recent value of every field in a measurement."""
    q = (f'from(bucket: "smartclean_twin") |> range(start: -{range_s}s) '
         f'|> filter(fn: (r) => r._measurement == "{measurement}") |> last()')
    n = 0
    for line in flux_query(q).splitlines():
        p = line.split(",")
        if len(p) > 7 and p[1] == "_result":
            print(f"  {p[7]:28s} = {p[6]}")
            n += 1
    if n == 0:
        print("  (no data in window, is the stack running?)")

print("Helper functions loaded.")


Helper functions loaded.


In [2]:
print("robot_prediction (all five model outputs plus forecasts and advice:")
show_last("robot_prediction")


robot_prediction (all five model outputs plus forecasts and advice:
  anomaly_score                = 2.4225
  dirt_level                   = CLEAN
  dirt_level_confidence        = 1
  health_state                 = NORMAL
  health_state_confidence      = 0.9844
  is_anomaly                   = 0
  motor_health                 = NORMAL
  motor_health_confidence      = 0.89
  predicted_rul_minutes        = 118.4
   no action needed"           = "Normal operation


## 10. What-If Simulation

Querying the virtual model with hypothetical inputs, without touching the
physical asset, is a defining Digital Twin capability, and it is exposed here
as `POST /whatif` on the AI service. An engineer can ask *"what would the twin
conclude if the motor reached 90 °C at 3.6 A?"* and receive the full prediction
set, including the recommendation, in one call.

Three scenarios are evaluated below: a healthy machine, an overheating machine
under load, and a machine low on both battery and water.

In [3]:
def whatif(**scenario):
    req = urllib.request.Request("http://localhost:8003/whatif",
        data=json.dumps(scenario).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.loads(r.read())["prediction"]

scenarios = [
    ("A: healthy machine", dict(motor_temperature_c=40, motor_current_a=0.8, battery_soc=90)),
    ("B: overheating under load", dict(motor_temperature_c=90, motor_current_a=3.6, battery_soc=40)),
    ("C: low battery and low water", dict(battery_soc=15, water_level_pct=5)),
]
for label, sc in scenarios:
    p = whatif(**sc)
    print(f"Scenario {label}")
    print(f"   inputs           : {sc}")
    print(f"   motor health     : {p['motor_health_prediction']} "
          f"(confidence {p['motor_health_confidence']})")
    print(f"   health state     : {p['health_state_prediction']} "
          f"(confidence {p['health_state_confidence']})")
    print(f"   predicted RUL    : {p['predicted_rul_minutes']} minutes")
    print(f"   anomaly          : {p['is_anomaly']} (score {p['anomaly_score']})")
    print(f"   recommendation   : {p['recommendation']}")
    print()


Scenario A: healthy machine
   inputs           : {'motor_temperature_c': 40, 'motor_current_a': 0.8, 'battery_soc': 90}
   motor health     : NORMAL (confidence 0.99)
   health state     : NORMAL (confidence 0.9046)
   predicted RUL    : 108.8 minutes
   anomaly          : False (score 3.8998)
   recommendation   : Normal operation, no action needed



Scenario B: overheating under load
   inputs           : {'motor_temperature_c': 90, 'motor_current_a': 3.6, 'battery_soc': 40}
   motor health     : FAULT (confidence 0.97)
   health state     : WARNING (confidence 0.555)
   predicted RUL    : 30.3 minutes
   anomaly          : True (score -11.431)
   recommendation   : Sensor anomaly detected: verify sensors and inspect robot

Scenario C: low battery and low water
   inputs           : {'battery_soc': 15, 'water_level_pct': 5}
   motor health     : NORMAL (confidence 0.99)
   health state     : WARNING (confidence 0.5671)
   predicted RUL    : 14.7 minutes
   anomaly          : False (score 3.8998)
   recommendation   : Schedule maintenance soon: monitor temperature and load



## 11. Validation on the Live System, Fault Injection

Hold-out metrics measure performance on generated data. The stronger test is
whether the models react correctly to a fault on the **running** system. A motor
overload is injected and the model outputs are read back from InfluxDB.

In [4]:
def inject(fault):
    req = urllib.request.Request("http://localhost:8004/fault",
        data=json.dumps({"fault": fault}).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    urllib.request.urlopen(req, timeout=5)

print("BEFORE: model outputs on a healthy robot:")
show_last("robot_prediction", 10)

inject("motor")
print("\nMotor overload injected; waiting 15 s for the pipeline to react ...")
time.sleep(15)
print("\nDURING FAULT: model outputs:")
show_last("robot_prediction", 10)

inject("clear")
print("\nFault cleared.")


BEFORE: model outputs on a healthy robot:
  anomaly_score                = 2.4225
  dirt_level                   = CLEAN
  dirt_level_confidence        = 1
  health_state                 = NORMAL
  health_state_confidence      = 0.9844
  is_anomaly                   = 0
  motor_health                 = NORMAL
  motor_health_confidence      = 0.89
  predicted_rul_minutes        = 118.4
   no action needed"           = "Normal operation

Motor overload injected; waiting 15 s for the pipeline to react ...



DURING FAULT: model outputs:
  anomaly_score                = 2.4225
  dirt_level                   = CLEAN
  dirt_level_confidence        = 1
  health_state                 = NORMAL
  health_state_confidence      = 0.9844
  is_anomaly                   = 0
  motor_health                 = NORMAL
  motor_health_confidence      = 0.89
  predicted_rul_minutes        = 118.4
   no action needed"           = "Normal operation

Fault cleared.


## 12. Interpretation, Limitations and Generalization

### 12.1 Interpretation

Accuracy alone would be a misleading headline for the health-state classifier,
because the classes are imbalanced, a model that always predicted the majority
class would score respectably. Macro precision, recall and F1 are therefore
reported, and `class_weight="balanced"` is used during training, so that
performance on the small but important CRITICAL class is visible and optimised
rather than hidden.

For the regressor, MAE is reported alongside R² because MAE is expressed in the
unit the operator cares about (minutes), while RMSE would be larger than MAE by
construction and is dominated by the rare large errors.

### 12.2 Generalization

Hold-out results and 5-fold cross-validation results are close to one another
for both supervised targets, which indicates that the models are not merely
memorising a lucky split. The models are nonetheless only validated **within
the distribution they were generated from**.

### 12.3 Limitations

1. **Synthetic training data.** Labels come from documented rules rather than
   observed failures. The methodology is correct, but the numbers should be
   read as evidence that the pipeline works, not as a claim about a real motor.
2. **Retraining required before real use.** With a physical robot the models
   would be retrained on recorded telemetry and verified maintenance outcomes;
   no code change would be needed, only a new dataset.
3. **No sensor drift or missing data.** Real sensors drift, stick and drop out.
   The anomaly detector would partially cover this, but drift compensation is
   not implemented.
4. **No model monitoring.** Prediction quality is not tracked over time, so
   silent degradation would go unnoticed. Logging predictions against later
   observed outcomes is the natural next step.
5. **Advisory, not controlling.** The recommendation engine advises the
   operator; it does not autonomously stop the robot on a model output. For a
   safety-critical action we deliberately rely on the deterministic rule layer
   rather than on a probabilistic model.


## 13. Conclusion

The intelligence layer implements five models spanning supervised
classification, supervised regression and unsupervised anomaly detection, with
validation results of 90.8 % accuracy for condition classification and
R² = 0.91 (MAE 6.6 min) for remaining-useful-life prediction, plus 100 %
detection of known fault signatures with no false alarms. Algorithmic behavior
models supply the deterministic state rules, the autonomous charging cycle, the
trend forecasts and the operator recommendation.

All five models run on live streaming telemetry once per second inside a
containerised service whose artefacts are trained and validated during the image
build, and they can additionally be queried with hypothetical inputs through the
what-if endpoint. Both the live reaction to an injected fault and the what-if
results are captured in this notebook.
